# 🔥 Apache Spark Workshop: DataFrames & Spark SQL

**Workshop Exercise | Estimated Time: 90 minutes**

---

In this exercise you will explore the fundamentals of Apache Spark using a **real-world retail dataset**. By the end you will understand:
- **Understand the SparkSession**: The entry point to Spark functionality.
- **Create and Inspect DataFrames, Schema Handling**: Inspecting, enforcing, and evolving schemas
- **Lazy Execution**: How Spark defers computation until an action is triggered
- **Transformations vs Actions**: Operations that build a plan vs those that execute it
- **Spark SQL**: Running SQL queries alongside DataFrame operations

### 📦 Dataset — Superstore Sales
A classic retail analytics dataset with **8,399 orders** across product categories, regions, and customer segments.
Originally published on Kaggle; we load it directly from a public GitHub mirror — no login required.

**Key columns we'll use:**

| Column | Type | Description |
|---|---|---|
| `Order ID` | string | Unique order identifier |
| `Order Date` | string → date | When the order was placed |
| `Ship Mode` | string | Shipping method |
| `Customer Name` | string | Customer |
| `Region` | string | Geographic region |
| `Product Category` | string | High-level category |
| `Product Sub-Category` | string | Detailed category |
| `Sales` | double | Revenue for the row |
| `Profit` | double | Profit for the row |
| `Order Quantity` | integer | Units ordered |
| `Discount` | double | Discount rate applied |
| `Ship Date` | string → date | Fulfilment date |

---
> **Tip**: Look for 🏋️ **Exercise** blocks throughout — those are your tasks to complete!
> There are Optional exercise which can be done at the end if the time permits

## Section 0 — Setup

### 0.1 Install dependencies (if needed)

Run the cell below only if PySpark is not already installed in your environment.

In [4]:
# Uncomment and run if pyspark is not installed
#!pip install pyspark --quiet
import sys
!{sys.executable} -m pip install pyspark --quiet

### 0.2 Start the SparkSession

Every Spark program begins with a `SparkSession` — the single entry point to DataFrames, SQL, and the cluster.

In [36]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType
)
from pyspark.sql.window import Window
import time

spark = SparkSession.builder \
    .appName("Spark Workshop — Superstore Sales") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"Cores available: {spark.sparkContext.defaultParallelism}")
print("SparkSession ready ✅")

Spark version : 4.1.1
Cores available: 12
SparkSession ready ✅


> **Tip**: Understand concept- "spark.sql.shuffle.partitions"

### 0.3 Download the dataset

We pull the CSV directly from a public GitHub repository — no Kaggle account or manual download required.

> **Dataset credit**: Sample Superstore Sales — widely used teaching dataset, publicly mirrored on GitHub.
> Direct link: https://raw.githubusercontent.com/BigDataGal/Python-for-Data-Science/master/Superstore-Sales.csv

In [37]:
import urllib.request
import os

DATA_URL  = "https://raw.githubusercontent.com/BigDataGal/Python-for-Data-Science/master/Superstore-Sales.csv"
DATA_PATH = "superstore_sales.csv"

if not os.path.exists(DATA_PATH):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"Saved to: {DATA_PATH}")
else:
    print(f"Dataset already present: {DATA_PATH}")

file_size_kb = os.path.getsize(DATA_PATH) / 1024
print(f"File size: {file_size_kb:.1f} KB")

Dataset already present: superstore_sales.csv
File size: 1727.7 KB


---
## Part 1 — Load Dataset,Schema Handling(Inferschema and explicit schema), Type Casting

### 1.1 Load with schema inference

The fastest way to get started: let Spark read the CSV and guess the schema automatically.

### 🏋️ Exercise 1.1 — Reading the CSV dataset using schema inference.
> **Tip**: using **"spark.read().option("inferSchema")"**

In [15]:
# Exercise 1 — your code here
# Load CSV with header and automatic schema inference
#df_inferred = 

#print(f"Rows: {df_inferred.count():,}")
#print(f"Columns: {len(df_inferred.columns)}\n")
#df_inferred.printSchema()

In [38]:
# ── Solution (run only after attempting!) ─────────────────────────────────
# Load CSV with header and automatic schema inference
df_inferred = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(DATA_PATH)

print(f"Rows: {df_inferred.count():,}")
print(f"Columns: {len(df_inferred.columns)}\n")
df_inferred.printSchema()

Rows: 8,399
Columns: 21

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: integer (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Order Priority: string (nullable = true)
 |-- Order Quantity: integer (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Province: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Product Container: string (nullable = true)
 |-- Product Base Margin: string (nullable = true)
 |-- Ship Date: string (nullable = true)



In [39]:
# Preview a few rows
df_inferred.select(
    "Order ID", "Order Date", "Region",
    "Product Category", "Sales", "Profit"
).show(5, truncate=False)

+--------+----------+-------+----------------+---------+-------+
|Order ID|Order Date|Region |Product Category|Sales    |Profit |
+--------+----------+-------+----------------+---------+-------+
|3       |10/13/2010|Nunavut|Office Supplies |261.54   |-213.25|
|293     |10/1/2012 |Nunavut|Office Supplies |10123.02 |457.81 |
|293     |10/1/2012 |Nunavut|Office Supplies |244.57   |46.71  |
|483     |7/10/2011 |Nunavut|Technology      |4965.7595|1198.97|
|515     |8/28/2010 |Nunavut|Office Supplies |394.27   |30.94  |
+--------+----------+-------+----------------+---------+-------+
only showing top 5 rows


**What to observe:**
- `Order Date` and `Ship Date` are inferred as `string` — Spark cannot automatically recognise `MM/dd/yyyy` as a date
- `Discount` is a `double` ✅ — numeric inference worked
- Column names have **spaces** (e.g. `Order ID`) — we'll rename them to snake_case

### 1.2 Define an explicit schema

In production you **always** define the schema explicitly:
- No expensive inference scan over the full file
- Types are guaranteed — no surprises at runtime
- Acts as inline documentation

In [40]:
explicit_schema = StructType([
    StructField("row_id",            IntegerType(), nullable=False),
    StructField("order_id",          IntegerType(), nullable=False),
    StructField("order_date",        StringType(),  nullable=True),   # cast to date below
    StructField("order_priority",    StringType(),  nullable=True),
    StructField("order_quantity",    IntegerType(), nullable=True),
    StructField("sales",             DoubleType(),  nullable=True),
    StructField("discount",          DoubleType(),  nullable=True),
    StructField("ship_mode",         StringType(),  nullable=True),
    StructField("profit",            DoubleType(),  nullable=True),
    StructField("unit_price",        DoubleType(),  nullable=True),
    StructField("shipping_cost",     DoubleType(),  nullable=True),
    StructField("customer_name",     StringType(),  nullable=True),
    StructField("province",          StringType(),  nullable=True),
    StructField("region",            StringType(),  nullable=True),
    StructField("customer_segment",  StringType(),  nullable=True),
    StructField("product_category",  StringType(),  nullable=True),
    StructField("product_sub_cat",   StringType(),  nullable=True),
    StructField("product_name",      StringType(),  nullable=True),
    StructField("product_container", StringType(),  nullable=True),
    StructField("product_base_margin", DoubleType(), nullable=True),
    StructField("ship_date",         StringType(),  nullable=True),   # cast to date below
])

# Load with explicit schema — no inferSchema scan
df_raw = spark.read \
    .option("header", True) \
    .schema(explicit_schema) \
    .csv(DATA_PATH)

print("=== Explicit Schema ===")
df_raw.printSchema()

=== Explicit Schema ===
root
 |-- row_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_priority: string (nullable = true)
 |-- order_quantity: integer (nullable = true)
 |-- sales: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- profit: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- province: string (nullable = true)
 |-- region: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_sub_cat: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_container: string (nullable = true)
 |-- product_base_margin: double (nullable = true)
 |-- ship_date: string (nullable = true)



### 1.3 Clean and enrich the DataFrame

- Cast date strings to `DateType` using spark functions: **to_date(column_name,format)**: "order_date", "ship_date"
- compute derived column "order_year", "order_month", "days_to_ship", "profit_margin" using **"df.withColumn()"**
- drop columns using **"drop(column_name)"** that will not be used in the exercise: "row_id", "product_container", "product_base_margin"

In [43]:

orders = df_raw \
    .withColumn("order_date", F.try_to_date("order_date", "M/d/yyyy")) \
    .withColumn("ship_date",  F.try_to_date("ship_date",  "M/d/yyyy")) \
    .withColumn("order_year",  F.year("order_date")) \
    .withColumn("order_month", F.month("order_date")) \
    .withColumn("days_to_ship",
        F.datediff(F.col("ship_date"), F.col("order_date"))) \
    .withColumn("profit_margin",
        F.round(F.col("profit") / F.col("sales"), 4)) \
    .drop("row_id", "product_container", "product_base_margin")

print("=== Final Schema ===")
orders.printSchema()

print(f"\nRows: {orders.count():,}  |  Columns: {len(orders.columns)}")
orders.show(3, truncate=True)

=== Final Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_priority: string (nullable = true)
 |-- order_quantity: integer (nullable = true)
 |-- sales: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- profit: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- province: string (nullable = true)
 |-- region: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_sub_cat: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- days_to_ship: integer (nullable = true)
 |-- profit_margin: double (nullable = true)


Rows: 8,399  |  Columns: 22
+

26/04/05 22:19:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order ID, Order Date, Order Priority, Order Quantity, Sales, Discount, Ship Mode, Profit, Unit Price, Shipping Cost, Customer Name, Province, Region, Customer Segment, Product Category, Product Sub-Category, Product Name, Ship Date
 Schema: order_id, order_date, order_priority, order_quantity, sales, discount, ship_mode, profit, unit_price, shipping_cost, customer_name, province, region, customer_segment, product_category, product_sub_cat, product_name, ship_date
Expected: order_id but found: Order ID
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


### 🏋️ Exercise 1 — Schema Exploration

Using the `orders` DataFrame, answer the following:

1. What is the data type of `order_date` now? And `profit`? (use `.dtypes`)
2. How many **null values** exist in the `product_base_margin` column — wait, we dropped it! Find the column with the most nulls among the remaining columns. (hint: `F.col(...).isNull()`)
3. Add a column called `is_profitable` that is `True` when `profit > 0`, `False` otherwise.
4. Show the min and max `order_date` in the dataset. (hint: `F.min()`, `F.max()` as actions via `.agg()`)

In [ ]:
# Exercise 1 — your code here

# 1. Data types
# YOUR CODE

# 2. Null counts per column
# YOUR CODE

# 3. Add is_profitable column
# orders = orders.withColumn(...)
# YOUR CODE

# 4. Date range
# YOUR CODE

In [44]:
# ── Solution (run only after attempting!) ─────────────────────────────────

# 1. Data types
print("=== Selected data types ===")
for name, dtype in orders.dtypes:
    if name in ("order_date", "profit", "ship_mode", "days_to_ship"):
        print(f"  {name}: {dtype}")

# 2. Null counts
print("\n=== Null counts per column ===")
null_counts = orders.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in orders.columns
])
null_counts.show(vertical=True)

# 3. is_profitable
orders = orders.withColumn("is_profitable", F.col("profit") > 0)
orders.select("order_id", "sales", "profit", "is_profitable").show(5)

# 4. Date range
print("=== Order date range ===")
orders.agg(
    F.min("order_date").alias("earliest_order"),
    F.max("order_date").alias("latest_order")
).show()

=== Selected data types ===
  order_date: date
  ship_mode: string
  profit: double
  days_to_ship: int

=== Null counts per column ===


26/04/05 22:19:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order ID, Order Date, Order Priority, Order Quantity, Sales, Discount, Ship Mode, Profit, Unit Price, Shipping Cost, Customer Name, Province, Region, Customer Segment, Product Category, Product Sub-Category, Product Name, Ship Date
 Schema: order_id, order_date, order_priority, order_quantity, sales, discount, ship_mode, profit, unit_price, shipping_cost, customer_name, province, region, customer_segment, product_category, product_sub_cat, product_name, ship_date
Expected: order_id but found: Order ID
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv
26/04/05 22:19:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order ID, Sales, Profit
 Schema: order_id, sales, profit
Expected: order_id but found: Order ID
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


-RECORD 0---------------
 order_id         | 0   
 order_date       | 0   
 order_priority   | 0   
 order_quantity   | 0   
 sales            | 0   
 discount         | 0   
 ship_mode        | 0   
 profit           | 0   
 unit_price       | 0   
 shipping_cost    | 0   
 customer_name    | 0   
 province         | 0   
 region           | 0   
 customer_segment | 0   
 product_category | 0   
 product_sub_cat  | 0   
 product_name     | 0   
 ship_date        | 290 
 order_year       | 0   
 order_month      | 0   
 days_to_ship     | 290 
 profit_margin    | 0   

+--------+---------+-------+-------------+
|order_id|    sales| profit|is_profitable|
+--------+---------+-------+-------------+
|       3|   261.54|-213.25|        false|
|     293| 10123.02| 457.81|         true|
|     293|   244.57|  46.71|         true|
|     483|4965.7595|1198.97|         true|
|     515|   394.27|  30.94|         true|
+--------+---------+-------+-------------+
only showing top 5 rows
=== Order dat

26/04/05 22:19:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order Date
 Schema: order_date
Expected: order_date but found: Order Date
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


---
## Part 2 — Lazy Execution

### 2.1 Understanding the execution model

Spark is **lazily evaluated**. When you call transformations (`.filter()`, `.select()`, `.withColumn()`), Spark does **not** execute them immediately. It builds a **logical plan** (a DAG — Directed Acyclic Graph). Execution only happens when you call an **action**.

```
Transformations  →  build a DAG (no data is read or moved)
Actions          →  trigger the DAG execution
```

In [45]:
# ── Timing demonstration ───────────────────────────────────────────────────

# Step 1: chain 4 transformations — should be near-instant
t0 = time.time()
high_value = (
    orders
    .filter(F.col("profit") > 0)                                    # transformation 1
    .filter(F.col("sales") > 500)                                   # transformation 2
    .withColumn("profit_pct", F.round(F.col("profit_margin") * 100, 1))  # transformation 3
    .select("customer_name", "region", "product_category",
            "sales", "profit", "profit_pct")                        # transformation 4
)
t1 = time.time()
print(f"Planning 4 transformations: {(t1 - t0) * 1000:.2f} ms  ← no data touched!")

# Step 2: trigger an action — Spark executes the full DAG NOW
t2 = time.time()
row_count = high_value.count()   # ACTION
t3 = time.time()
print(f"Executing .count() action:  {(t3 - t2) * 1000:.1f} ms  ← real computation")
print(f"Matching rows: {row_count:,}")

Planning 4 transformations: 47.04 ms  ← no data touched!
Executing .count() action:  58.0 ms  ← real computation
Matching rows: 2,637


### 2.2 Reading the execution plan

`.explain()` shows what Spark *plans* to do before executing it — essential for debugging performance.

In [46]:
# Simple plan — just the physical steps
print("=== Simple Plan ===")
high_value.explain(mode="simple")

=== Simple Plan ===
== Physical Plan ==
*(1) Project [customer_name#1412, region#1414, product_category#1416, sales#1406, profit#1409, round((round((profit#1409 / sales#1406), 4) * 100.0), 1) AS profit_pct#2151]
+- *(1) Filter (((isnotnull(profit#1409) AND isnotnull(sales#1406)) AND (profit#1409 > 0.0)) AND (sales#1406 > 500.0))
   +- FileScan csv [sales#1406,profit#1409,customer_name#1412,region#1414,product_category#1416] Batched: false, DataFilters: [isnotnull(profit#1409), isnotnull(sales#1406), (profit#1409 > 0.0), (sales#1406 > 500.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ku50kw/Developer/superstore_sales.csv], PartitionFilters: [], PushedFilters: [IsNotNull(profit), IsNotNull(sales), GreaterThan(profit,0.0), GreaterThan(sales,500.0)], ReadSchema: struct<sales:double,profit:double,customer_name:string,region:string,product_category:string>




In [47]:
# Extended plan — logical + optimised + physical
print("=== Extended Plan ===")
high_value.explain(mode="extended")

=== Extended Plan ===
== Parsed Logical Plan ==
'Project ['customer_name, 'region, 'product_category, 'sales, 'profit, 'profit_pct]
+- Project [order_id#1402, order_date#1757, order_priority#1404, order_quantity#1405, sales#1406, discount#1407, ship_mode#1408, profit#1409, unit_price#1410, shipping_cost#1411, customer_name#1412, province#1413, region#1414, customer_segment#1415, product_category#1416, product_sub_cat#1417, product_name#1418, ship_date#1758, order_year#1759, order_month#1760, days_to_ship#1761, profit_margin#1762, is_profitable#2091, round((profit_margin#1762 * cast(100 as double)), 1) AS profit_pct#2151]
   +- Filter (sales#1406 > cast(500 as double))
      +- Filter (profit#1409 > cast(0 as double))
         +- Project [order_id#1402, order_date#1757, order_priority#1404, order_quantity#1405, sales#1406, discount#1407, ship_mode#1408, profit#1409, unit_price#1410, shipping_cost#1411, customer_name#1412, province#1413, region#1414, customer_segment#1415, product_catego

**Key things to notice:**
- `Filter` nodes may be **pushed down** — Spark applies them early so it reads less data
- `Project` = column selection / `withColumn`
- The optimised logical plan may look different from the code you wrote — Spark's Catalyst optimiser rewrites it

### 🏋️ Exercise 2 — Lazy Execution

1. Build a chain of **at least 3 transformations** on `orders` (your choice of operations: Eg Put filters on Column)
2. Call `.explain(mode="simple")` — read and understand the plan
3. Time the transformation chain vs an action (use `time.time()`)

In [ ]:
# Exercise 2 — your code here

# 1. Chain of 3+ transformations
# my_df = orders. ...

# 2. Inspect the plan
# my_df.explain(mode="simple")

# 3. Time transformations vs action

# 4. Bonus: cache + compare two .count() calls

In [48]:
# ── Solution (run only after attempting!) ─────────────────────────────────

# 1. Chain of 3 transformations — LAZY, no execution yet
t0 = time.time()
my_df = (
    orders
    .filter(F.col("region") == "West")                              # transformation 1
    .filter(F.col("product_category") == "Technology")             # transformation 2
    .withColumn("sales_k", F.round(F.col("sales") / 1000, 3))     # transformation 3
    .select("customer_name", "product_sub_cat", "sales", "sales_k", "profit")
)
t1 = time.time()
print(f"Planning time:  {(t1 - t0) * 1000:.2f} ms  ← zero computation")

# 2. Execution plan
print("\n=== Execution Plan ===")
my_df.explain(mode="simple")
# Notice: both Filter nodes appear — Spark may merge or push them down
# Notice: Project corresponds to our .select()

# 3. Time the action
t2 = time.time()
my_df.show(5)                                                       # ACTION
t3 = time.time()
print(f"\nFirst .show() action: {(t3 - t2) * 1000:.1f} ms")


Planning time:  44.38 ms  ← zero computation

=== Execution Plan ===
== Physical Plan ==
*(1) Project [customer_name#1412, product_sub_cat#1417, sales#1406, round((sales#1406 / 1000.0), 3) AS sales_k#2164, profit#1409]
+- *(1) Filter (((isnotnull(region#1414) AND isnotnull(product_category#1416)) AND (region#1414 = West)) AND (product_category#1416 = Technology))
   +- FileScan csv [sales#1406,profit#1409,customer_name#1412,region#1414,product_category#1416,product_sub_cat#1417] Batched: false, DataFilters: [isnotnull(region#1414), isnotnull(product_category#1416), (region#1414 = West), (product_categor..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ku50kw/Developer/superstore_sales.csv], PartitionFilters: [], PushedFilters: [IsNotNull(region), IsNotNull(product_category), EqualTo(region,West), EqualTo(product_category,T..., ReadSchema: struct<sales:double,profit:double,customer_name:string,region:string,product_category:string,prod...


+-----------------+----------

26/04/05 22:22:31 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Sales, Profit, Customer Name, Region, Product Category, Product Sub-Category
 Schema: sales, profit, customer_name, region, product_category, product_sub_cat
Expected: customer_name but found: Customer Name
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


---
## Part 3 — Transformations vs Actions

### 3.1 Common transformations

Transformations return a **new DataFrame** and are **lazy** — they never trigger computation.

In [49]:
# ── select: choose columns ────────────────────────────────────────────────
orders.select("customer_name", "region", "product_category", "sales", "profit").show(4)

+------------------+-------+----------------+---------+-------+
|     customer_name| region|product_category|    sales| profit|
+------------------+-------+----------------+---------+-------+
|Muhammed MacIntyre|Nunavut| Office Supplies|   261.54|-213.25|
|      Barry French|Nunavut| Office Supplies| 10123.02| 457.81|
|      Barry French|Nunavut| Office Supplies|   244.57|  46.71|
|     Clay Rozendal|Nunavut|      Technology|4965.7595|1198.97|
+------------------+-------+----------------+---------+-------+
only showing top 4 rows


26/04/05 22:22:47 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Sales, Profit, Customer Name, Region, Product Category
 Schema: sales, profit, customer_name, region, product_category
Expected: customer_name but found: Customer Name
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


In [50]:
# ── filter: keep rows matching a condition ────────────────────────────────
furniture = orders.filter(F.col("product_category") == "Furniture")
print(f"Furniture orders: {furniture.count():,}")
furniture.show(3)

Furniture orders: 1,724
+--------+----------+--------------+--------------+-------+--------+--------------+------+----------+-------------+---------------+--------+-------+----------------+----------------+------------------+--------------------+----------+----------+-----------+------------+-------------+-------------+
|order_id|order_date|order_priority|order_quantity|  sales|discount|     ship_mode|profit|unit_price|shipping_cost|  customer_name|province| region|customer_segment|product_category|   product_sub_cat|        product_name| ship_date|order_year|order_month|days_to_ship|profit_margin|is_profitable|
+--------+----------+--------------+--------------+-------+--------+--------------+------+----------+-------------+---------------+--------+-------+----------------+----------------+------------------+--------------------+----------+----------+-----------+------------+-------------+-------------+
|     515|2010-08-28| Not Specified|            21| 146.69|    0.05|   Regular Air

26/04/05 22:22:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Product Category
 Schema: product_category
Expected: product_category but found: Product Category
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv
26/04/05 22:22:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order ID, Order Date, Order Priority, Order Quantity, Sales, Discount, Ship Mode, Profit, Unit Price, Shipping Cost, Customer Name, Province, Region, Customer Segment, Product Category, Product Sub-Category, Product Name, Ship Date
 Schema: order_id, order_date, order_priority, order_quantity, sales, discount, ship_mode, profit, unit_price, shipping_cost, customer_name, province, region, customer_segment, product_category, product_sub_cat, product_name, ship_date
Expected: order_id but found: Order ID
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


In [51]:
# ── withColumn: add or replace a column ──────────────────────────────────
orders_enriched = orders.withColumn(
    "ship_speed",
    F.when(F.col("days_to_ship") <= 1, "Same/Next Day")
     .when(F.col("days_to_ship") <= 4, "Express")
     .otherwise("Standard")
)
orders_enriched.select("order_id", "ship_mode", "days_to_ship", "ship_speed").show(6)

+--------+--------------+------------+-------------+
|order_id|     ship_mode|days_to_ship|   ship_speed|
+--------+--------------+------------+-------------+
|       3|   Regular Air|           7|     Standard|
|     293|Delivery Truck|           1|Same/Next Day|
|     293|   Regular Air|           2|      Express|
|     483|   Regular Air|           2|      Express|
|     515|   Regular Air|           2|      Express|
|     515|   Regular Air|           2|      Express|
+--------+--------------+------------+-------------+
only showing top 6 rows


26/04/05 22:22:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order ID, Order Date, Ship Mode, Ship Date
 Schema: order_id, order_date, ship_mode, ship_date
Expected: order_id but found: Order ID
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


In [52]:
# ── groupBy + agg: aggregate ──────────────────────────────────────────────
category_summary = orders.groupBy("product_category").agg(
    F.count("order_id").alias("num_orders"),
    F.round(F.sum("sales"), 2).alias("total_sales"),
    F.round(F.sum("profit"), 2).alias("total_profit"),
    F.round(F.avg("discount") * 100, 1).alias("avg_discount_pct")
)
category_summary.orderBy(F.col("total_sales").desc()).show()

+----------------+----------+-----------+------------+----------------+
|product_category|num_orders|total_sales|total_profit|avg_discount_pct|
+----------------+----------+-----------+------------+----------------+
|      Technology|      2065| 5984248.18|   886313.52|             4.9|
|       Furniture|      1724| 5178590.54|   117433.03|             4.9|
| Office Supplies|      4610|  3752762.1|   518021.43|             5.0|
+----------------+----------+-----------+------------+----------------+



26/04/05 22:23:02 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order ID, Sales, Discount, Profit, Product Category
 Schema: order_id, sales, discount, profit, product_category
Expected: order_id but found: Order ID
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


In [53]:
# ── join: combine two DataFrames ──────────────────────────────────────────
# Create a small region-tier lookup
region_tiers = spark.createDataFrame([
    ("West",    "Tier 1"),
    ("East",    "Tier 1"),
    ("Central", "Tier 2"),
    ("South",   "Tier 2"),
], ["region", "tier"])

orders_tiered = orders.join(region_tiers, on="region", how="left")
orders_tiered.select("order_id", "region", "tier", "sales").show(5)

+--------+-------+----+---------+
|order_id| region|tier|    sales|
+--------+-------+----+---------+
|       3|Nunavut|NULL|   261.54|
|     293|Nunavut|NULL| 10123.02|
|     293|Nunavut|NULL|   244.57|
|     483|Nunavut|NULL|4965.7595|
|     515|Nunavut|NULL|   394.27|
+--------+-------+----+---------+
only showing top 5 rows


26/04/05 22:23:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order ID, Sales, Region
 Schema: order_id, sales, region
Expected: order_id but found: Order ID
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


In [54]:
# ── window function: rank within a group ──────────────────────────────────
window_cat = Window.partitionBy("product_category").orderBy(F.col("sales").desc())

orders \
    .withColumn("rank_in_category", F.rank().over(window_cat)) \
    .filter(F.col("rank_in_category") == 1) \
    .select("product_category", "customer_name", "product_name", "sales") \
    .orderBy("product_category") \
    .show(truncate=True)

+----------------+--------------+--------------------+--------+
|product_category| customer_name|        product_name|   sales|
+----------------+--------------+--------------------+--------+
|       Furniture| Steve Chapman|Riverside Palais ...|29345.27|
| Office Supplies|John Stevenson|Fellowes PB500 El...|25409.63|
|      Technology|    Emily Phan|Polycom ViewStati...|89061.05|
+----------------+--------------+--------------------+--------+



26/04/05 22:23:06 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Sales, Customer Name, Product Category, Product Name
 Schema: sales, customer_name, product_category, product_name
Expected: customer_name but found: Customer Name
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv


### 3.2 Common actions

Actions **trigger execution** of the DAG and return a result to the driver (or write to storage).

In [55]:
# count — full scan to count rows
print("Total orders:", orders.count())

# first — fetch one row to driver
row = orders.first()
print("\nFirst row order_id:", row["order_id"])

# show — print N rows in console
orders.select("customer_name", "sales", "profit").show(3)

# collect — bring ALL rows to driver (⚠️ dangerous on large data!)
west_tech = orders.filter(
    (F.col("region") == "West") & (F.col("product_category") == "Technology")
).select("customer_name", "sales").collect()
print(f"\nWest Technology orders collected: {len(west_tech)} rows")

# take — collect first N rows to driver
top5 = orders.orderBy(F.col("sales").desc()).take(5)
print("\nTop 5 by sales:")
for r in top5:
    print(f"  {r['customer_name']:<22} ${r['sales']:>10,.2f}")

Total orders: 8399

First row order_id: 3
+------------------+--------+-------+
|     customer_name|   sales| profit|
+------------------+--------+-------+
|Muhammed MacIntyre|  261.54|-213.25|
|      Barry French|10123.02| 457.81|
|      Barry French|  244.57|  46.71|
+------------------+--------+-------+
only showing top 3 rows

West Technology orders collected: 507 rows

Top 5 by sales:
  Emily Phan             $ 89,061.05
  Jasper Cacioppo        $ 45,923.76
  Craig Carreira         $ 41,343.21
  Dennis Kane            $ 33,367.85
  Karen Carlisle         $ 29,884.60


26/04/05 22:23:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order ID, Order Date, Order Priority, Order Quantity, Sales, Discount, Ship Mode, Profit, Unit Price, Shipping Cost, Customer Name, Province, Region, Customer Segment, Product Category, Product Sub-Category, Product Name, Ship Date
 Schema: order_id, order_date, order_priority, order_quantity, sales, discount, ship_mode, profit, unit_price, shipping_cost, customer_name, province, region, customer_segment, product_category, product_sub_cat, product_name, ship_date
Expected: order_id but found: Order ID
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv
26/04/05 22:23:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Sales, Profit, Customer Name
 Schema: sales, profit, customer_name
Expected: customer_name but found: Customer Name
CSV file: file:///Users/ku50kw/Developer/superstore_sales.csv
26/04/05 22:23:11 WARN CSVHeaderChecker: CSV header does not confor

### 🏋️ Exercise 3 — Transformations & Actions

Using the `orders` DataFrame, build a pipeline that answers:

> **"Which are the top 5 most profitable sub-categories in the West region, and what is their average discount?"**

Requirements:
1. **Filter** to the `West` region only
2. **Group by** `product_sub_cat`
3. **Aggregate**: total profit, number of orders, avg discount (as %)
4. **Add a column** `profit_label`: `"High"` if total_profit > 5000, else `"Normal"`
5. **Sort** by total profit descending and show the top 5
6. Trigger with **`.show(5)`**

In [ ]:
# Exercise 3 — your code here

# result = orders \
#     .filter(...) \
#     .groupBy(...).agg(...) \
#     .withColumn("profit_label", ...) \
#     .orderBy(...)

# result.show(5)

In [ ]:
# ── Solution ───────────────────────────────────────────────────────────────
result = orders \
    .filter(F.col("region") == "West") \
    .groupBy("product_sub_cat").agg(
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.count("order_id").alias("num_orders"),
        F.round(F.avg("discount") * 100, 1).alias("avg_discount_pct")
    ) \
    .withColumn(
        "profit_label",
        F.when(F.col("total_profit") > 5000, "High").otherwise("Normal")
    ) \
    .orderBy(F.col("total_profit").desc())

result.show(5, truncate=False)

---
## Part 4 — Spark SQL

Spark SQL lets you query DataFrames using standard SQL syntax. You first register a DataFrame as a **temporary view** — it lives for the duration of the SparkSession.

### 4.1 Register views

In [ ]:
# Register the main orders DataFrame
orders.createOrReplaceTempView("orders")
region_tiers.createOrReplaceTempView("region_tiers")

# Verify
spark.sql("SELECT COUNT(*) AS total_rows FROM orders").show()
spark.sql("SELECT * FROM orders LIMIT 3").show(truncate=True)

### 4.2 Basic queries

In [ ]:
# Simple SELECT with WHERE and ORDER BY
spark.sql("""
    SELECT
        customer_name,
        region,
        product_category,
        ROUND(sales, 2)  AS sales,
        ROUND(profit, 2) AS profit
    FROM orders
    WHERE profit > 1000
      AND region = 'West'
    ORDER BY profit DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# GROUP BY with multiple aggregations
spark.sql("""
    SELECT
        region,
        product_category,
        COUNT(*)                          AS num_orders,
        ROUND(SUM(sales), 2)              AS total_sales,
        ROUND(SUM(profit), 2)             AS total_profit,
        ROUND(AVG(discount) * 100, 1)     AS avg_discount_pct
    FROM orders
    GROUP BY region, product_category
    ORDER BY total_profit DESC
    LIMIT 12
""").show()

In [ ]:
# JOIN with the region_tiers lookup
spark.sql("""
    SELECT
        o.region,
        t.tier,
        COUNT(*)                AS num_orders,
        ROUND(SUM(o.sales), 2)  AS total_sales,
        ROUND(SUM(o.profit), 2) AS total_profit
    FROM orders o
    LEFT JOIN region_tiers t ON o.region = t.region
    GROUP BY o.region, t.tier
    ORDER BY total_profit DESC
""").show()

In [ ]:
# Window function: rank sub-categories by profit within each category
spark.sql("""
    SELECT
        product_category,
        product_sub_cat,
        ROUND(SUM(profit), 2) AS total_profit,
        RANK() OVER (
            PARTITION BY product_category
            ORDER BY SUM(profit) DESC
        ) AS profit_rank
    FROM orders
    GROUP BY product_category, product_sub_cat
    ORDER BY product_category, profit_rank
""").show(20)

### 4.3 SQL ↔ DataFrame interoperability

A SQL result is just a DataFrame — you can chain DataFrame operations after `spark.sql()`.

In [ ]:
# SQL result → continue with DataFrame API
annual = spark.sql("""
    SELECT
        order_year,
        ROUND(SUM(sales), 2)  AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit
    FROM orders
    GROUP BY order_year
""")

# Chain DataFrame operations on top of the SQL result
annual \
    .withColumn("margin_pct",
        F.round(F.col("total_profit") / F.col("total_sales") * 100, 1)) \
    .orderBy("order_year") \
    .show()

### 🏋️ Exercise 4 — Spark SQL

Write SQL queries to answer the following business questions:

**Q1.** What are the **top 3 customers by total sales** in each region? (use a window function)

**Q2.** Which **ship mode** has the highest average profit margin (`profit / sales`)? Exclude rows where `sales = 0`.

**Q3.** Show **monthly sales and profit trends** for the year with the most orders. (hint: find the year with `COUNT(*)`, then use it in a subquery or hardcode it after Q1)

**Bonus Q4.** Find all **sub-categories where total profit is negative** — these are loss-making product lines. Show category, sub-category, total sales, and total profit.

In [ ]:
# Q1 — Top 3 customers by sales per region
spark.sql("""
    -- YOUR SQL HERE
""").show()

In [ ]:
# Q2 — Ship mode with highest avg profit margin
spark.sql("""
    -- YOUR SQL HERE
""").show()

In [ ]:
# Q3 — Monthly trends in the busiest year
spark.sql("""
    -- YOUR SQL HERE
""").show()

In [ ]:
# Bonus Q4 — Loss-making sub-categories
spark.sql("""
    -- YOUR SQL HERE
""").show()

In [ ]:
# ── Solutions ──────────────────────────────────────────────────────────────

# Q1
print("Q1 — Top 3 customers by sales per region:")
spark.sql("""
    SELECT region, customer_name, total_sales, sales_rank
    FROM (
        SELECT
            region,
            customer_name,
            ROUND(SUM(sales), 2) AS total_sales,
            RANK() OVER (PARTITION BY region ORDER BY SUM(sales) DESC) AS sales_rank
        FROM orders
        GROUP BY region, customer_name
    )
    WHERE sales_rank <= 3
    ORDER BY region, sales_rank
""").show(20)

# Q2
print("Q2 — Ship mode by avg profit margin:")
spark.sql("""
    SELECT
        ship_mode,
        ROUND(AVG(profit / sales) * 100, 2) AS avg_margin_pct,
        COUNT(*) AS num_orders
    FROM orders
    WHERE sales > 0
    GROUP BY ship_mode
    ORDER BY avg_margin_pct DESC
""").show()

# Q3
print("Q3 — Monthly trends in the busiest year:")
spark.sql("""
    SELECT
        order_month,
        COUNT(*)                 AS num_orders,
        ROUND(SUM(sales), 2)     AS monthly_sales,
        ROUND(SUM(profit), 2)    AS monthly_profit
    FROM orders
    WHERE order_year = (
        SELECT order_year FROM orders
        GROUP BY order_year
        ORDER BY COUNT(*) DESC
        LIMIT 1
    )
    GROUP BY order_month
    ORDER BY order_month
""").show()

# Bonus Q4
print("Q4 — Loss-making sub-categories:")
spark.sql("""
    SELECT
        product_category,
        product_sub_cat,
        ROUND(SUM(sales), 2)  AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit
    FROM orders
    GROUP BY product_category, product_sub_cat
    HAVING SUM(profit) < 0
    ORDER BY total_profit ASC
""").show()

---
## Part 5 — Capstone Challenge 🏆

Put everything together using **both the DataFrame API and Spark SQL**.

### Task — Executive Sales Dashboard

Build a report that shows, **for each region and customer segment**:
- Total orders and total sales
- Total profit and profit margin %
- Average days to ship
- % of orders that are profitable (`profit > 0`)
- A `performance` label: `"Strong"` (margin > 15%), `"Moderate"` (> 0%), `"Loss-making"` otherwise
- Overall rank by profit (across all region+segment combinations)

**Requirements:**
- Use at least one **Spark SQL** query
- Use at least one **DataFrame transformation** (e.g. `withColumn`, `join`)
- Use a **window function** for the overall ranking
- Final result sorted by rank

In [ ]:
# Capstone — your code here

# Step 1: SQL aggregation
# base = spark.sql("""...""")

# Step 2: DataFrame transformations (labels, window rank)
# ...

# Step 3: Show final result
# dashboard.show(truncate=False)

In [ ]:
# ── Capstone Solution ──────────────────────────────────────────────────────

# Step 1: Aggregate via SQL
base = spark.sql("""
    SELECT
        region,
        customer_segment,
        COUNT(*)                            AS total_orders,
        ROUND(SUM(sales), 2)                AS total_sales,
        ROUND(SUM(profit), 2)               AS total_profit,
        ROUND(AVG(days_to_ship), 1)         AS avg_days_to_ship,
        ROUND(SUM(CASE WHEN profit > 0 THEN 1 ELSE 0 END)
              / COUNT(*) * 100, 1)          AS pct_profitable_orders
    FROM orders
    GROUP BY region, customer_segment
""")

# Step 2: DataFrame API — add margin, performance label, and ranking
window_spec = Window.orderBy(F.col("total_profit").desc())

dashboard = base \
    .withColumn("margin_pct",
        F.round(F.col("total_profit") / F.col("total_sales") * 100, 1)) \
    .withColumn("performance",
        F.when(F.col("margin_pct") > 15, "Strong")
         .when(F.col("margin_pct") > 0,  "Moderate")
         .otherwise("Loss-making")) \
    .withColumn("rank", F.rank().over(window_spec)) \
    .select(
        "rank", "region", "customer_segment",
        "total_orders", "total_sales", "total_profit",
        "margin_pct", "avg_days_to_ship",
        "pct_profitable_orders", "performance"
    ) \
    .orderBy("rank")

print("=== Executive Sales Dashboard ===")
dashboard.show(20, truncate=False)

---
## Recap & Key Takeaways

| Concept | What you learned |
|---|---|
| **Schema Handling** | Inferred vs explicit schemas; `StructType`; casting dates; null inspection |
| **Lazy Execution** | Transformations build a DAG; actions execute it; `.explain()` reads the plan; `.cache()` avoids re-execution |
| **Transformations** | `select`, `filter`, `withColumn`, `groupBy().agg()`, `join`, `orderBy`, `F.when()` |
| **Actions** | `count`, `show`, `collect`, `first`, `take` — each re-executes the DAG unless cached |
| **Spark SQL** | `createOrReplaceTempView`, `spark.sql()`, window functions, `HAVING`, subqueries |
| **Interoperability** | SQL results are DataFrames; mix both APIs freely |

### Further reading
- [PySpark DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)
- [Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [Spark Performance Tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html)
- [Superstore dataset on Kaggle](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

In [ ]:
spark.stop()
print("SparkSession stopped. Workshop complete! 🎉")